In [1]:
import json
from transformers import pipeline
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
from rouge import Rouge
from scipy.stats import ttest_rel, wilcoxon

Открытие размеченного с помощью Label Studio json-файла и преобразование его в удобный для работы json-файл, исключая избыточную информацию

In [2]:
with open("../data/project-9-at-2025-05-14-14-05-d436104d.json", "r") as f:
    data = json.load(f)

formatted_data = []
for item in data:
    text = item["data"]["text"]
    keywords = item["annotations"][0]["result"][0]["value"]["text"][0].split(", ")
    formatted_data.append({"text": text, "keywords": keywords})

with open("../data/formatted_data.json", "w") as f:
    json.dump(formatted_data, f, indent=2)

Открытие json-файла для работы

In [3]:
with open("../data/formatted_data.json", "r") as f:
    formatted_data = json.load(f)

bart_model = pipeline("text2text-generation", model="ilsilfverskiold/bart-keyword-extractor")
tech_model = pipeline("text2text-generation", model="ilsilfverskiold/tech-keywords-extractor")

Device set to use cuda:0
Device set to use cuda:0


Функция для оценки метрик (precision, recall, f1, rouge) качества предсказания ключевых слов с помощью выбранных моделей

In [4]:
def evaluate_model(model, data, compute_rouge=True):

    mlb = MultiLabelBinarizer()
    rouge = Rouge() if compute_rouge else None
    all_true_keywords = []
    all_pred_keywords = []
    pred_texts = []
    true_texts = []
    
    for item in data:

        pred = model(item["text"], max_length=50)[0]["generated_text"]
        pred_keywords = [kw.strip() for kw in pred.split(",")]
        true_keywords = item["keywords"]
        
        all_true_keywords.append(true_keywords)
        all_pred_keywords.append(pred_keywords)
        
        if compute_rouge:
            pred_texts.append(pred)
            true_texts.append(", ".join(true_keywords))
    
    mlb.fit(all_true_keywords + all_pred_keywords)
    true_binary = mlb.transform(all_true_keywords)
    pred_binary = mlb.transform(all_pred_keywords)
    
    metrics = {
        "precision": precision_score(true_binary, pred_binary, average='micro'),
        "recall": recall_score(true_binary, pred_binary, average='micro'),
        "f1": f1_score(true_binary, pred_binary, average='micro')
    }
    
    if compute_rouge:
        rouge_scores = rouge.get_scores(pred_texts, true_texts, avg=True)
        metrics.update({
            "rouge-1": rouge_scores["rouge-1"]["f"],
            "rouge-2": rouge_scores["rouge-2"]["f"],
            "rouge-l": rouge_scores["rouge-l"]["f"]
        })
    
    return metrics

In [5]:
print("Bart_model: \n", evaluate_model(bart_model, formatted_data))
print("Tech_model: \n", evaluate_model(tech_model, formatted_data))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Bart_model: 
 {'precision': 0.2886435331230284, 'recall': 0.32562277580071175, 'f1': 0.3060200668896321, 'rouge-1': 0.34105782236194454, 'rouge-2': 0.14796749032833606, 'rouge-l': 0.324283384442647}
Tech_model: 
 {'precision': 0.05787781350482315, 'recall': 0.04804270462633452, 'f1': 0.05250364608653379, 'rouge-1': 0.05078286132940366, 'rouge-2': 0.01348464605917546, 'rouge-l': 0.05078286132940366}


Функция для оценки метрик (precision, recall, f1) качества предсказания ключевых слов с помощью выбранных моделей. Функция на выходе дает как усредненную оценку метрик качества (mean_metrics), так и отдельные значения метрик для каждого вопроса, из которого будут извлекаться ключевые слова (sample metrics). Эти отдельные метрики качества предсказания необходимы в дальнейшем для статистического сравнения результатов предсказания ключевых слов разными моделями.

In [6]:
def evaluate_model_per_sample(model, data):
    mlb = MultiLabelBinarizer()
    all_true_keywords = []
    all_pred_keywords = []
    
    for item in data:
        pred = model(item["text"], max_length=50)[0]["generated_text"]
        pred_keywords = [kw.strip() for kw in pred.split(",")]
        all_true_keywords.append(item["keywords"])
        all_pred_keywords.append(pred_keywords)
    
    mlb.fit(all_true_keywords + all_pred_keywords)
    true_binary = mlb.transform(all_true_keywords)
    pred_binary = mlb.transform(all_pred_keywords)
    
    mean_metrics = {
        "precision": precision_score(true_binary, pred_binary, average='micro'),
        "recall": recall_score(true_binary, pred_binary, average='micro'),
        "f1": f1_score(true_binary, pred_binary, average='micro')
    }
    
    sample_metrics = []
    for item in data:
        pred = model(item["text"], max_length=50)[0]["generated_text"]
        pred_keywords = [kw.strip() for kw in pred.split(",")]
        true_keywords = item["keywords"]
        
        true_binary = mlb.transform([true_keywords])
        pred_binary = mlb.transform([pred_keywords])
        
        sample_metrics.append({
            "precision": precision_score(true_binary, pred_binary, average='micro'),
            "recall": recall_score(true_binary, pred_binary, average='micro'),
            "f1": f1_score(true_binary, pred_binary, average='micro')
        })
    
    return mean_metrics, sample_metrics

Выполним оценку метрик качества предсказания с помощью выбранных моделей. После этого проведем статистическое сравнение результатов, полученных для 2 выбранных моделей. Для статистического сравнения выбраны 2 критерия: t-критерий (предполагая нормальность распределения различий между значениями метрик для 2 моделей) и критерий Уилкоксона (отсутствие предположения о нормальности). Кроме того, для t-критерия следует учитывать, что наши выборки (т.е. набор вопросов) являются зависимыми (это одни и те же вопросы, оцениваемые обеими моделями). Поэтому будет использовать ttest_rel критерий, а не ttest_ind критерий. Для критерия Уилкоксона зависимость выборок считается нормальным фактором.

In [7]:
mean_metrics_bart_model, metrics_bart_model = evaluate_model_per_sample(bart_model, formatted_data)
mean_metrics_tech_model, metrics_tech_model = evaluate_model_per_sample(tech_model, formatted_data)

print("Bart_model averaged metrics:\n", mean_metrics_bart_model)
print("Tech_model averaged metrics:\n", mean_metrics_tech_model)

f1_bart_model = [m["f1"] for m in metrics_bart_model]
f1_tech_model = [m["f1"] for m in metrics_tech_model]

# t-критерий
t_stat, p_value = ttest_rel(f1_bart_model, f1_tech_model)
print(f"\nt-тест: p-value = {p_value:.4f}")

# Критерий Уилкоксона
wilcoxon_stat, wilcoxon_p = wilcoxon(f1_bart_model, f1_tech_model)
print(f"\nТест Уилкоксона: p-value = {wilcoxon_p:.4f}")

Bart_model averaged metrics:
 {'precision': 0.2886435331230284, 'recall': 0.32562277580071175, 'f1': 0.3060200668896321}
Tech_model averaged metrics:
 {'precision': 0.05787781350482315, 'recall': 0.04804270462633452, 'f1': 0.05250364608653379}

t-тест: p-value = 0.0000

Тест Уилкоксона: p-value = 0.0000


Заключение

И t-критерий, и критерий Уилкоксона демонстрируют, что между средними значениями показателей f1, найденных с помощью двух моделей, существует статистически достоверное различие. Поскольку модель bart_model демонстрирует более высокие результаты, выберем ее для дальнейшего использования в проекте.